In [1]:
# Mrinal Pradhan
# HW5
# Least Squares Method 

import numpy as np
import matplotlib.pyplot as plt

def least_squares_custom(x, y):
    # Fit y = m*x + b using closed-form least squares
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    xMean = x.mean()
    yMean = y.mean()

    denominator = np.sum((x - xMean) ** 2)
    if denominator == 0:
        raise ValueError("All x values are identical, cannot fit a line.")

    m = np.sum((x - xMean) * (y - yMean)) / denominator
    b = yMean - m * xMean

    yHat = m * x + b
    residuals = y - yHat
    return m, b, residuals, yHat

# Tests method usig random vectors of X and Y

rng = np.random.default_rng(123)
n = 30
Xr = rng.random(n)
Yr = rng.random(n)

# sort X and reorder Y in the same order to preserve pairing
idx = np.argsort(Xr)
Xr = Xr[idx]
Yr = Yr[idx]

m_c, b_c, res_c, yhat_c = least_squares_custom(Xr, Yr)

A = np.vstack([Xr, np.ones(len(Xr))]).T
m_n, b_n = np.linalg.lstsq(A, Yr, rcond=None)[0]
yhat_n = m_n * Xr + b_n
res_n = Yr - yhat_n

print("Random X,Y test")
print(f"Custom: m={m_c:.8f}, b={b_c:.8f}")
print(f"Numpy : m={m_n:.8f}, b={b_n:.8f}")
print(f"Differences: |dm|={abs(m_c-m_n):.3e}, |db|={abs(b_c-b_n):.3e}")

# Residual extremes (custom)
i_abs_max = np.argmax(np.abs(res_c))
i_abs_min = np.argmin(np.abs(res_c))

print("\nResiduals:")
print(f"Sum of squared residuals: {np.sum(res_c**2):.8f}")
print(f"Largest |res|: idx={i_abs_max}, x={Xr[i_abs_max]:.3f}, y={Yr[i_abs_max]:.3f}, res={res_c[i_abs_max]:.6f}")
print(f"Smallest |res|: idx={i_abs_min}, x={Xr[i_abs_min]:.3f}, y={Yr[i_abs_min]:.3f}, res={res_c[i_abs_min]:.6f}")

# Plot random test
xline = np.linspace(0, 1, 300)
plt.figure()
plt.scatter(Xr, Yr, s=25, label="Points")
plt.plot(xline, m_c * xline + b_c, label="Custom fit")
plt.plot(xline, m_n * xline + b_n, linestyle="--", label="Numpy fit")
plt.title("Random X,Y test")
plt.xlabel("X")
plt.ylabel("Y")
plt.legend()
plt.show()

# 2) Random linear function with random error between 0 and 1
#    y = m*x + b +/- random_number(0,1)
#    X: 0 to 1 step 0.01

m_true = rng.uniform(-5, 5)
b_true = rng.uniform(-2, 2)

X = np.arange(0.0, 1.0001, 0.01)

# +/- random in [0,1]
sign = rng.choice([-1.0, 1.0], size=len(X))
noise = rng.random(len(X)) * sign
Y = m_true * X + b_true + noise

m_c2, b_c2, res_c2, yhat_c2 = least_squares_custom(X, Y)

A2 = np.vstack([X, np.ones(len(X))]).T
m_n2, b_n2 = np.linalg.lstsq(A2, Y, rcond=None)[0]
yhat_n2 = m_n2 * X + b_n2
res_n2 = Y - yhat_n2

print("\nRandom linear function")
print(f"True  : m={m_true:.8f}, b={b_true:.8f}")
print(f"Custom: m={m_c2:.8f}, b={b_c2:.8f}")
print(f"Numpy : m={m_n2:.8f}, b={b_n2:.8f}")
print(f"Differences: |dm|={abs(m_c2-m_n2):.3e}, |db|={abs(b_c2-b_n2):.3e}")

# Residual extremes 
i_abs_max2 = np.argmax(np.abs(res_c2))
i_abs_min2 = np.argmin(np.abs(res_c2))

print("\nResiduals: ")
print(f"Sum of squared residuals: {np.sum(res_c2**2):.8f}")
print(f"Largest |res|: idx={i_abs_max2}, x={X[i_abs_max2]:.2f}, y={Y[i_abs_max2]:.3f}, yhat={yhat_c2[i_abs_max2]:.3f}, res={res_c2[i_abs_max2]:.6f}")
print(f"Smallest |res|: idx={i_abs_min2}, x={X[i_abs_min2]:.2f}, y={Y[i_abs_min2]:.3f}, yhat={yhat_c2[i_abs_min2]:.3f}, res={res_c2[i_abs_min2]:.6f}")

# Plot 
xline2 = np.linspace(0, 1, 400)
plt.figure()
plt.scatter(X, Y, s=12, label="Points (noisy)")
plt.plot(xline2, m_c2 * xline2 + b_c2, label="Custom best-fit")
plt.plot(xline2, m_true * xline2 + b_true, linestyle="--", label="True line")
plt.title("Random Linear Function")
plt.xlabel("X")
plt.ylabel("Y")
plt.legend()
plt.show()
# For the Random X,Y test the slope is the exact same for both lines and for the random linear function test the slopes are nearly the same.
# This shows that the custom implementation correctly reproduces the same solution that NumPy computes.

Random X,Y test
Custom: m=-0.19939506, b=0.56107735
Numpy : m=-0.19939506, b=0.56107735
Differences: |dm|=8.327e-17, |db|=1.110e-16

Residuals:
Sum of squared residuals: 2.41145374
Largest |res|: idx=19, x=0.682, y=0.917, res=0.492103
Smallest |res|: idx=15, x=0.518, y=0.468, res=0.010555


<Figure size 640x480 with 1 Axes>


Random linear function
True  : m=-0.00513250, b=1.03524128
Custom: m=0.07073362, b=0.88124385
Numpy : m=0.07073362, b=0.88124385
Differences: |dm|=3.886e-16, |db|=2.220e-16

Residuals: 
Sum of squared residuals: 29.33430927
Largest |res|: idx=24, x=0.24, y=2.016, yhat=0.898, res=1.117407
Smallest |res|: idx=51, x=0.51, y=0.915, yhat=0.917, res=-0.002771


<Figure size 640x480 with 1 Axes>